# 抽象網格題：庫存與現金的關係（APCS 風格）
Status: in_use  
Prereq: list/dict、函式、模組化基本、能獨立跑小專題  
Can-do checklist:  
- [ ] 能用網格/狀態表示題目
- [ ] 能用迴圈模擬狀態更新
- [ ] 能設計並驗證邊界條件


本題把網格邏輯抽象成一個純狀態機問題：
- `inventory` 代表庫存
- `cash` 代表現金
- 價格只會上下穿越固定網格線

重點是：**庫存與現金會同時變動，但結算規則非常簡單**。


## 必要知識點（邏輯概念）
1. **狀態機**：用 `position / inventory / cash` 描述當下狀態。
2. **穿越判斷**：是否跨過某條線，決定是否觸發買賣。
3. **模擬**：按序列一步一步更新狀態，不需要公式。
4. **現金與庫存關係**：
   - 向下穿越 → 庫存 +1，現金 -price
   - 向上穿越且庫存 >0 → 庫存 -1，現金 +price


## 題目敘述（APCS 風格）
一個點在數線上移動，數線上有固定網格線。

給定：
- 區間 `L ~ R`
- 間距 `d`（網格線是 L+d, L+2d, ..., R-d）
- 起點 `x0`
- 一串移動序列 `moves`
- 初始現金 `cash0`

規則：
1. 若點向下穿越一條線（prev > line 且 curr <= line），執行：
   - inventory += 1
   - cash -= line
2. 若點向上穿越一條線（prev < line 且 curr >= line），且 inventory > 0，執行：
   - inventory -= 1
   - cash += line

請輸出：
- 最後位置
- 最後庫存 inventory
- 最後現金 cash
- 總資產值 = cash + inventory * 當前位置


### 輸入格式
第一行：L R d
第二行：x0 cash0
第三行：n（移動次數）
第四行：n 個整數 moves

### 輸出格式
輸出四個整數或浮點數：
position inventory cash total


### 範例輸入
```
0 10 2
5 100
8
1 1 -1 -1 -1 2 1 -2
```

### 範例輸出（其中一種）
```
5 1 98 103
```
（實際輸出依照規則逐步計算即可）


## 解題思路（抽象化）
1. 建立所有網格線
2. 逐步模擬移動
3. 每次移動都檢查是否跨越多條線
4. 按規則更新 inventory 與 cash

這是一題 **純模擬題**，沒有捷徑。


In [ ]:
def lines_crossed(prev, curr, lines):
    if curr > prev:
        return [line for line in lines if prev < line <= curr]
    if curr < prev:
        return [line for line in reversed(lines) if curr <= line < prev]
    return []


In [ ]:
def simulate(L, R, d, x0, cash0, moves):
    lines = list(range(L + d, R, d))
    position = x0
    inventory = 0
    cash = cash0

    for move in moves:
        prev = position
        position += move

        for line in lines_crossed(prev, position, lines):
            if prev > position:  # 向下
                inventory += 1
                cash -= line
            else:  # 向上
                if inventory > 0:
                    inventory -= 1
                    cash += line

    total = cash + inventory * position
    return position, inventory, cash, total


## 範例驗證
（用範例輸入測試）


In [ ]:
L, R, d = 0, 10, 2
x0, cash0 = 5, 100
moves = [1, 1, -1, -1, -1, 2, 1, -2]
simulate(L, R, d, x0, cash0, moves)


## 觀念整理：庫存與現金的關係
- 庫存增加時，現金必定減少（以當下穿越線的價格計算）。
- 庫存減少時，現金必定增加。
- 若位置回到起點，現金可能仍然增加，因為中間完成過買賣配對。

這正是網格的核心邏輯，但在本題完全抽象成狀態更新問題。


## 進階延伸題
1. 加入庫存上限 cap：超過就不能再買。
2. 加入手續費 fee：每次買賣都扣固定比例。
3. 若 moves 可能跨越區間外，是否要忽略或直接停止？
